# 전체 학습 과정 및 아키텍쳐

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%pip install --upgrade --quiet \
    langchain \
    langchain-core \
    langchain-community \
    langchain-openai \
    langchain-experimental \
    langchain-neo4j \
    neo4j \
    python-dotenv \
    tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.0/234.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7

In [4]:
from google.colab import userdata

NEO4J_URI=userdata.get('NEO4J_URI')
NEO4J_USERNAME=userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD=userdata.get('NEO4J_PASSWORD')
NEO4J_DATABASE=userdata.get('NEO4J_DATABASE')
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [5]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

load_dotenv()

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

# 연결 확인
result = graph.query("RETURN 1 AS ping")
print(result)  # [{'ping': 1}] 이면 성공

[{'ping': 1}]


In [6]:
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings
import os

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",   # 저렴하고 성능 충분
    openai_api_key=OPENAI_API_KEY
)

vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    index_name="entity_vector_index",
    node_label="__Entity__",           # Phase 2에서 baseEntityLabel=True로 생성된 라벨
    text_node_properties=["id", "description"],       # 임베딩할 텍스트 속성
    embedding_node_property="embedding",
    search_type="hybrid",              # vector + keyword 동시 활용
)

print("벡터 인덱스 생성 완료")


벡터 인덱스 생성 완료


In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",      # 추출용으로 가성비가 좋은 모델
    temperature=0,
    openai_api_key=OPENAI_API_KEY
)

### 3차 시도

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# 뉴스 요약을 위한 간단한 프롬프트
news_summary_prompt = ChatPromptTemplate.from_template(
    "다음 뉴스 원문을 SCM(공급망) 리스크 관점에서 1~2문장으로 핵심만 요약해줘. 관련 링크가 있다면 포함해.\n\n원문: {text}"
)
news_summary_chain = news_summary_prompt | llm | StrOutputParser()


def graph_retriever(question: str) -> str:
    question_vector = embeddings.embed_query(question)

    query = """
    CALL db.index.vector.queryNodes('entity_vector_index', 5, $embedding)
    YIELD node AS n, score

    MATCH path = (n)-[r*1..2]-(neighbor)
    WHERE NOT neighbor:Document
      AND ALL(rel IN r WHERE type(rel) <> 'MENTIONS')

    OPTIONAL MATCH (n)-[:MENTIONS]-(doc:Document)

    RETURN n.id AS source,
           n.description AS source_desc,
           [rel IN r | type(rel)] AS relations,
           neighbor.id AS target,
           neighbor.description AS target_desc,
           labels(neighbor)[0] AS target_type,
           collect(DISTINCT {text: doc.text, link: doc.link}) AS related_news,
           score
    ORDER BY score DESC
    LIMIT 15
    """

    response = graph.query(query, params={"embedding": question_vector})
    if not response: return "관련 정보 없음"

    results = []
    seen_entities = set()
    processed_news = {} # 뉴스 요약 재사용을 위한 캐시

    for row in response:
        # 1. 핵심 엔티티 및 뉴스 요약 처리
        if row['source'] not in seen_entities:
            s_desc = row['source_desc'] or "설명 없음"
            results.append(f"\n[핵심 엔티티] {row['source']} ({s_desc})")

            if row['related_news']:
                for news in row['related_news']:
                    if news['text']:
                        # 이미 요약한 뉴스가 아니라면 요약 수행
                        if news['text'] not in processed_news:
                            summary = news_summary_chain.invoke({"text": news['text']})
                            processed_news[news['text']] = summary

                        results.append(f"  📰 뉴스 요약: {processed_news[news['text']]}")
                        if news['link']:
                            results.append(f"  🔗 출처: {news['link']}")

            seen_entities.add(row['source'])

        # 2. 전파 경로
        rel_path = " -> ".join(row['relations'])
        results.append(f"  - 경로: {row['source']} -[{rel_path}]-> {row['target']} ({row['target_type']})")

        # 3. 타겟 정보
        if row['target'] not in seen_entities:
            t_desc = row['target_desc'] or "상세 정보 없음"
            results.append(f"  - 영향 대상: {row['target']} ({t_desc})")
            seen_entities.add(row['target'])

    return "\n".join(results)


# 3. LLM 프롬프트 및 체인 설정
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 현대자동차 기업의 SCM(공급망 관리) 분석 전문가입니다.
제공된 '참고 정보'는 지식 그래프에서 추출된 엔티티 간의 관계입니다.

이 정보를 바탕으로 질문에 대해 논리적이고 전문적으로 답변하십시오.
만약 정보가 부족하다면 추측하기보다 정보가 부족함을 알리세요.

""")

    ,
    ("human", "질문: {question}\n\n참고 정보:\n{context}"),
])

qa_chain = qa_prompt | llm | StrOutputParser()


In [9]:
# 4. 실행 및 결과 출력
question = "안전 공업의 엔진 생산을 대체할만한 공급처가 있을까?"
context = graph_retriever(question)

print("--- [Retrieved Context from Graph] ---")
print(context)

answer = qa_chain.invoke({"question": question, "context": context})

print("\n--- [최종 AI SCM 분석 답변] ---")
print(answer)

--- [Retrieved Context from Graph] ---

[핵심 엔티티] 엔진 생산 차질 (설명 없음)
  📰 뉴스 요약: 부품업체의 화재로 인해 현대차와 기아의 엔진 생산 라인이 중단되며 공급망에 차질이 발생하고 있습니다. 이 사고는 협력사의 문제로 인해 전체 생산에 영향을 미치는 공급망 리스크를 보여줍니다.
  🔗 출처: https://www.econovill.com/news/articleView.html?idxno=734980
  - 경로: 엔진 생산 차질 -[ADOPTS]-> 현대자동차 (__Entity__)
  - 영향 대상: 현대자동차 (상세 정보 없음)
  - 경로: 엔진 생산 차질 -[ADOPTS -> CAUSES]-> 울산대교 (__Entity__)
  - 영향 대상: 울산대교 (상세 정보 없음)
  - 경로: 엔진 생산 차질 -[ADOPTS -> CAUSES]-> 산업단지 (__Entity__)
  - 영향 대상: 산업단지 (상세 정보 없음)
  - 경로: 엔진 생산 차질 -[ADOPTS -> ADDRESSES]-> 유럽 드론 공급망 진입 기회 (__Entity__)
  - 영향 대상: 유럽 드론 공급망 진입 기회 (상세 정보 없음)
  - 경로: 엔진 생산 차질 -[ADOPTS -> ADDRESSES]-> 사외이사 제도 도입 (__Entity__)
  - 영향 대상: 사외이사 제도 도입 (상세 정보 없음)
  - 경로: 엔진 생산 차질 -[ADOPTS -> ADDRESSES]-> 174억6100만 원 (__Entity__)
  - 영향 대상: 174억6100만 원 (상세 정보 없음)
  - 경로: 엔진 생산 차질 -[ADOPTS -> ADDRESSES]-> 90억100만 원 (__Entity__)
  - 영향 대상: 90억100만 원 (상세 정보 없음)
  - 경로: 엔진 생산 차질 -[ADOPTS -> ADDRESSES]-> 54억 원 (__Entity__)
  - 영향 대상: 54억 원 (상세 정보 없음)
  - 경로: 엔

In [10]:
# 4. 실행 및 결과 출력
question = "중동 전쟁이 현대자동차 공급망에 미치는 영향은?"
context = graph_retriever(question)

print("--- [Retrieved Context from Graph] ---")
print(context)

answer = qa_chain.invoke({"question": question, "context": context})

print("\n--- [최종 AI SCM 분석 답변] ---")
print(answer)

--- [Retrieved Context from Graph] ---

[핵심 엔티티] 자동차 산업 중심의 수출 공급망 강화 프로그램 (설명 없음)
  📰 뉴스 요약: 은행권이 가계대출 규제로 인해 기업금융으로 전환하면서, 하나은행은 현대자동차기아 및 HL만도와 협력하여 자동차 산업의 수출 공급망 강화 프로그램을 운영하고 있습니다. 이는 기업의 거래 구조에 맞춘 유연한 자금 공급을 통해 산업 생태계 전반의 리스크를 줄이려는 노력으로 볼 수 있습니다.
  🔗 출처: https://www.newsfreezone.co.kr/news/articleView.html?idxno=682461
  - 경로: 자동차 산업 중심의 수출 공급망 강화 프로그램 -[ADDRESSES]-> 경쟁 리스크 (__Entity__)
  - 영향 대상: 경쟁 리스크 (상세 정보 없음)
  - 경로: 자동차 산업 중심의 수출 공급망 강화 프로그램 -[ADDRESSES -> COMPETES_WITH]-> 현대자동차 (__Entity__)
  - 영향 대상: 현대자동차 (상세 정보 없음)
  - 경로: 자동차 산업 중심의 수출 공급망 강화 프로그램 -[ADDRESSES -> COMPETES_WITH]-> Byd (__Entity__)
  - 영향 대상: Byd (상세 정보 없음)
  - 경로: 자동차 산업 중심의 수출 공급망 강화 프로그램 -[ADDRESSES -> COMPETES_WITH]-> 구글 (__Entity__)
  - 영향 대상: 구글 (상세 정보 없음)
  - 경로: 자동차 산업 중심의 수출 공급망 강화 프로그램 -[ADDRESSES -> COMPETES_WITH]-> 삼성전자 (__Entity__)
  - 영향 대상: 삼성전자 (상세 정보 없음)
  - 경로: 자동차 산업 중심의 수출 공급망 강화 프로그램 -[ADDRESSES -> COMPETES_WITH]-> Sk하이닉스 (__Entity__)
  - 영향 대상: Sk하이닉스 (상세 정보 없음)
  - 경로: 자동차 산업 중심의 